### Goal of notebook: merge RNA raw count matrices between libraries
- Takes output from /u/project/cluo/terencew/igvf/2023_YR2/multiome/scripts/freeze_barcodes/A01b_run_rna_subset.sh 
- Output from previous script: cells by features matrix, subset to ambimux singlets
- Add library identifier for each barcode

In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd

import scanpy as sc
import anndata

from scipy.io import mmwrite

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))

In [ ]:
### load matrices with scanpy and merge together
s = samples[0]
mtx_path = f'{projdir}/rna/ambimux/wgs/{s}/'
adata = sc.read_10x_mtx(mtx_path, cache=False)
adata.var_names_make_unique()
adata.obs.index = [f'{x}_{s}' for x in adata.obs.index]

for s in samples[1:]:
    print(s)
    mtx_path = f'{projdir}/rna/ambimux/wgs/{s}/'
    tmp_adata = sc.read_10x_mtx(mtx_path, cache=False)
    tmp_adata.var_names_make_unique()
    tmp_adata.obs.index = [f'{x}_{s}' for x in tmp_adata.obs.index]
    adata = anndata.concat([adata, tmp_adata])

In [ ]:
adata.obs.head()

In [ ]:
barcodes_out = f'{projdir}/mtx/ambimux/wgs/merged/barcodes.tsv.gz'
adata.obs.to_csv(barcodes_out, sep='\t', header=False, index=True, compression='gzip')

### write genes out

In [ ]:
s = samples[0]
genes_path = f'{projdir}/rna/ambimux/wgs/{s}/features.tsv.gz'
genes = pd.read_csv(genes_path, sep='\t', header=None, index_col=None)
mask = ['ENSG' in x for x in genes[0]]
tmp_genes = genes[mask]
tmp_genes.shape

In [ ]:
tmp_peaks = genes[~np.array(mask)]
tmp_peaks.shape

In [ ]:
tmp_peaks

In [ ]:
genes_out = f'{projdir}/mtx/ambimux/wgs/merged/features.tsv.gz'
tmp_genes.to_csv(genes_out, sep='\t', header=False, index=False, compression='gzip')

In [ ]:
adata.shape

In [ ]:
adata.X.shape

In [ ]:
raw_out = f'{projdir}/mtx/ambimux/wgs/merged/matrix.mtx'
mmwrite(raw_out, adata.X)

In [ ]:
adata

In [ ]:
ambimux = pd.read_csv(f'{projdir}/csv/demux/ambimux/wgs/ambimux_joint_sings.csv', sep='\t', header=0, index_col=0)
ambimux.shape

In [ ]:
### check if all the demultiplexed barcodes are there
ambimux = ambimux.reindex(adata.obs.index)

np.sum(ambimux.index.isin(adata.obs.index)), \
np.sum(adata.obs.index.isin(ambimux.index)), \
np.sum(ambimux.index == adata.obs.index)

### generate QC metrics for filtering

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
adata.var['mt'].value_counts()

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

In [ ]:
adata.var.head()

In [ ]:
adata.write_h5ad(f'{projdir}/h5ad/rna/raw/ambimux/wgs/merged.h5ad')